# 13 — Residence time of Stagnone lagoon water

**Research question:** how long does water stay inside the lagoon?

**Approach** (literature brief in `memory/project_residence_time.md`):
1. **Primary method — Eulerian passive tracer:** set tracer `C=1` inside lagoon, `C=0` at open boundary (Dirichlet), integrate; extract per-cell e-folding time and bulk τ from the decay curve. Standard in Mar Menor (García-Oliva 2019), Venice (Cucco & Umgiesser 2006), Sacca di Goro (Ferrarin 2013), Ria Formosa (Dias 2009/2013).
2. **Sanity check — Knudsen salt balance:** for hypersaline lagoons, τ ≈ V·ΔS / (E·S_lag) where V = lagoon volume, ΔS = salinity excess over offshore, E = evaporation flux. Bulk number only but independent physics.
3. **Cross-check — Lagrangian** (notebook 32_analysis_particle_tracking framework, seed lagoon domain, measure 50/90% exit time).
4. **Future extension — CART age tracer** (Deleersnijder/Delhez): two coupled user tracers for age field. Gold standard but requires custom tracer setup.

**Status:** this notebook computes the Knudsen bulk estimate **now** (from v02 + ERA5) and preps the passive-tracer setup + post-processing for the next model run (**v03+**, which will also carry the corrected hypersaline initial field — see `memory/feedback_salinity_bc.md`).

**Ensemble reminder** (Viero & Defina 2016): a single run's residence time is meaningless in a wind-driven lagoon. Eventually run the tracer under ≥3 forcing scenarios (summer sirocco, winter mistral, calm) and report the distribution.

## 1. Imports and paths

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath

project_root = Path(r'F:\StagnoneDT')
v02_dir = project_root / 'model' / 'dflowfm_v02'
v02_map = v02_dir / 'output' / 'Stagnone_dxy01_15m_map.nc'
processed_dir = project_root / 'data' / 'processed'
era5_dir = v02_dir  # ERA5 meteo files live alongside v02 inputs

print(f'v02 map file exists: {v02_map.exists()}')

## 2. Load lagoon interior polygon

Authoritative polygon: boundary extracted from the `oldModel/Stagnone_justLagoon` mesh (hand-drawn around the lagoon proper), reprojected from EPSG:3857 to WGS84. 82-vertex simplified version in `data/processed/lagoon_polygon_wgs84_simplified.csv`.

Drives: (a) cell mask for V / ΔS; (b) initial tracer field; (c) masked post-processing average.

In [ ]:
poly_csv = processed_dir / 'lagoon_polygon_wgs84_simplified.csv'
poly_df = pd.read_csv(poly_csv)
lagoon_poly = poly_df[['lon', 'lat']].values
lagoon_path = MplPath(lagoon_poly)

print(f'Polygon: {len(lagoon_poly)} vertices')
print(f'Lon range: {lagoon_poly[:,0].min():.4f} .. {lagoon_poly[:,0].max():.4f}')
print(f'Lat range: {lagoon_poly[:,1].min():.4f} .. {lagoon_poly[:,1].max():.4f}')

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(lagoon_poly[:, 0], lagoon_poly[:, 1], '-', color='tab:blue', lw=0.8)
ax.fill(lagoon_poly[:, 0], lagoon_poly[:, 1], alpha=0.15)
ax.set_aspect(1 / np.cos(np.radians(37.87)))
ax.set_xlabel('Lon'); ax.set_ylabel('Lat'); ax.grid(alpha=0.3)
ax.set_title('Lagoon polygon (from Stagnone_justLagoon mesh)')
plt.tight_layout(); plt.show()

## 3. Lagoon volume from v02 bathymetry

Volume needed for the Knudsen estimate. Use v02 mesh cell areas × time-averaged water depth for interior cells only.

In [ ]:
uds = xu.open_dataset(str(v02_map), chunks={'time': 50})
face_coords = uds.grid.face_coordinates
face_x = np.asarray(face_coords[:, 0])
face_y = np.asarray(face_coords[:, 1])

# Identify interior cells via polygon containment (face-center test)
pts = np.column_stack([face_x, face_y])
inside = lagoon_path.contains_points(pts)
print(f'Interior cells: {inside.sum()} / {len(inside)}  ({100 * inside.mean():.1f}% of mesh)')

In [ ]:
# Cell areas: use xugrid's internal area accessor if available, else compute from face nodes
try:
    face_area = uds.grid.area  # m² (UGRID convention for projected, may need conversion for WGS84)
    face_area = np.asarray(face_area)
except Exception as e:
    print(f'xugrid area accessor failed ({e}); falling back to shoelace on lat/lon converted to m.')
    # Rough lat/lon → m conversion at lagoon lat, then shoelace formula per face
    face_area = None  # TODO: implement if needed

# Time-averaged water depth per face
wd_mean = uds['mesh2d_waterdepth'].mean(dim='time').compute().values

interior_area_m2 = face_area[inside].sum() if face_area is not None else np.nan
interior_vol_m3 = (face_area[inside] * wd_mean[inside]).sum() if face_area is not None else np.nan
mean_depth_m = wd_mean[inside].mean()

print(f'Lagoon interior area ≈ {interior_area_m2 / 1e6:.2f} km² (lit. ~12 km²)')
print(f'Mean depth inside   ≈ {mean_depth_m:.2f} m (lit. 0.5–2 m)')
print(f'Volume estimate     ≈ {interior_vol_m3 / 1e6:.2f} ×10⁶ m³')

## 4. Evaporation flux from ERA5

ERA5 provides evaporation `e` (m of water per time step, negative for upward flux). We need the area-mean evaporation rate over the lagoon during the simulation period, converted to m/s and then to a volume loss rate (m³/s).

In [ ]:
era5_e = sorted(era5_dir.glob('era5_e_*.nc'))
if not era5_e:
    print('No era5_e_*.nc found — evaporation was not included in v02 meteo.')
    print('Either download ERA5 `e` with dfm_tools or fall back to a literature value.')
    # Mediterranean summer evaporation: ~5-7 mm/day ≈ 6e-8 to 8e-8 m/s
    E_ms_literature = 6e-8   # m/s, July Mediterranean estimate
    print(f'Using literature value: {E_ms_literature:.1e} m/s ≈ {E_ms_literature * 86400 * 1000:.1f} mm/day')
    E_ms = E_ms_literature
else:
    ds_e = xr.open_mfdataset([str(f) for f in era5_e])
    # ERA5 `e` is accumulated over the accumulation period; convert to m/s mean
    print('Opened ERA5 evaporation files:', [f.name for f in era5_e])
    # Placeholder: actual conversion depends on variable name and accumulation — check ds_e vars
    print(ds_e.data_vars)
    E_ms = None  # to be set after inspecting the file

if E_ms is not None:
    E_m3s = E_ms * interior_area_m2
    print(f'Evaporative volume loss ≈ {E_m3s:.2f} m³/s over {interior_area_m2/1e6:.1f} km²')

## 5. Knudsen residence-time estimate

For a quasi-steady hypersaline lagoon the salt balance gives

$$\tau \;\approx\; \frac{V \, \Delta S}{E \, S_{\text{lag}}}$$

where `V` is lagoon volume, `ΔS = S_lag − S_offshore`, `E` is evaporative volume flux, `S_lag` is the interior salinity.

With Stagnone's target values (S_lag ≈ 42, S_offshore ≈ 37.5, ΔS ≈ 4.5 psu) this is a first-order check. Valid only if salt storage is near steady — fails during transient freshening events.

In [ ]:
S_lag = 42.0       # psu, literature for Stagnone interior
S_off = 37.5       # psu, offshore Mediterranean
dS = S_lag - S_off

if (interior_vol_m3 is not np.nan) and (E_ms is not None):
    tau_s = (interior_vol_m3 * dS) / (E_ms * interior_area_m2 * S_lag)
    tau_days = tau_s / 86400
    print(f'Knudsen residence time estimate: {tau_days:.1f} days ({tau_days/30:.1f} months)')
    print(f'  V       = {interior_vol_m3/1e6:.1f} ×10⁶ m³')
    print(f'  ΔS/S    = {dS/S_lag:.3f}')
    print(f'  E (vol) = {E_ms * interior_area_m2:.2f} m³/s')
else:
    print('Inputs incomplete; run cells 3–4 successfully first.')

## 6. Passive-tracer setup recipe for v03+

To enable the Eulerian passive tracer in the next D-Flow FM run:

**a. Declare tracer in `Stagnone_dxy01_15m.mdu`** under `[transport]`:
```
TransportMethod = 1           # advection-diffusion
TransportTimestepping = 1
```
and under `[tracers]` (custom user tracer):
```
TracerNames = lagoon_tracer
```

**b. Initial field via `initialFields.ini`** — attach a tracer init block pointing to an XYZ file with `1.0` inside the lagoon polygon, `0.0` outside:
```
[Initial]
    quantity = initialtracerlagoon_tracer
    dataFile = lagoon_tracer_init.xyz
    dataFileType = sample
    interpolationMethod = averaging
    averagingType = mean
    operand = O
```

**c. Open boundary Dirichlet=0** in the `.ext` file, one block per boundary segment:
```
[Boundary]
    quantity = tracerbndlagoon_tracer
    locationFile = Stagnone_dxy01_15m.pli
    forcingFile  = tracer_zero.bc
```
with `tracer_zero.bc` a constant-0 forcing.

The cell below writes the XYZ sample file for the interior mask (step a).

In [ ]:
# Build lagoon_tracer_init.xyz from the polygon mask.
# Use face centers; inside=1.0, outside=0.0. D-Flow FM will nearest-average onto the mesh.
tracer_vals = np.where(inside, 1.0, 0.0)
v03_dir = project_root / 'model' / 'dflowfm_v03'
v03_dir.mkdir(parents=True, exist_ok=True)
xyz_path = v03_dir / 'lagoon_tracer_init.xyz'

with open(xyz_path, 'w') as f:
    for x, y, v in zip(face_x, face_y, tracer_vals):
        f.write(f'{x:.6f} {y:.6f} {v:.3f}\n')

print(f'Wrote {xyz_path}  ({inside.sum()} cells = 1.0, {(~inside).sum()} cells = 0.0)')

# And the zero-tracer BC file
bc_path = v03_dir / 'tracer_zero.bc'
with open(bc_path, 'w') as f:
    f.write('[General]\nfileVersion = 1.01\nfileType = boundConds\n\n'
            '[Forcing]\nname = Stagnone_dxy01_15m_bnd1_0001\n'
            'function = constant\nquantity = tracerbndlagoon_tracer\nunit = -\n0.0\n')
print(f'Wrote {bc_path}')

## 7. Post-processing the tracer run (v03 output)

Once v03 finishes, the `_map.nc` will carry `mesh2d_lagoon_tracer(time, face)`. From that we compute:

- **Bulk decay curve:** area-weighted mean of the tracer inside the lagoon polygon vs time
- **Bulk residence time τ:** fit `C(t) = exp(-t/τ)` to the decay, take τ as the e-folding timescale
- **Spatial residence-time map:** fit exponential per cell, plot τ(x,y)
- **50% / 90% flushing times:** first time the mean drops below 0.5 and 0.1

Skeleton below — activate after v03 run.

In [ ]:
def fit_efolding(t_s, c):
    """Fit C(t) = exp(-t/τ) in log-space. Returns τ in seconds, or NaN if fit fails."""
    c = np.asarray(c, dtype=float)
    valid = (c > 0.01) & np.isfinite(c)
    if valid.sum() < 3:
        return np.nan
    slope, _ = np.polyfit(t_s[valid], np.log(c[valid]), 1)
    return -1.0 / slope if slope < 0 else np.nan

# --- skeleton, to run on v03 output ---
# v03_map = project_root / 'model' / 'dflowfm_v03' / 'output' / 'Stagnone_dxy01_15m_map.nc'
# uds3 = xu.open_dataset(str(v03_map), chunks={'time': 50})
# tracer = uds3['mesh2d_lagoon_tracer']          # (time, face)
# times = pd.to_datetime(tracer.time.values)
# t_s = (times - times[0]).total_seconds()
#
# # bulk decay curve (area-weighted mean over interior)
# w = face_area[inside]
# C_t = ((tracer.isel(face=np.where(inside)[0]) * w).sum('face') / w.sum()).compute().values
# tau_bulk_s = fit_efolding(t_s, C_t)
# print(f'Bulk e-folding residence time: {tau_bulk_s/86400:.2f} days')
#
# t50 = t_s[np.argmax(C_t < 0.5)] / 86400 if (C_t < 0.5).any() else np.nan
# t90 = t_s[np.argmax(C_t < 0.1)] / 86400 if (C_t < 0.1).any() else np.nan
# print(f'50% flushing: {t50:.2f} days   90% flushing: {t90:.2f} days')
#
# # per-cell e-folding map
# tau_cell = xr.apply_ufunc(
#     lambda c: fit_efolding(t_s, c),
#     tracer, input_core_dims=[['time']], vectorize=True,
# ).compute()
print('Skeleton ready; activate once v03 map.nc is produced.')

## 8. Cross-check: Lagrangian 50/90 exit time

Using notebook 32_analysis_particle_tracking's framework, seed a dense grid of particles inside the lagoon polygon (e.g. 500–2000), run for ~30+ days, and measure the time for 50% / 90% of particles to leave the polygon. Should match the tracer e-folding τ within a factor ~1.5 (differences stem from re-entry handling and diffusivity choice).

This is a separate execution (add to notebook 32_analysis_particle_tracking or create `14_residence_lagrangian.ipynb`).

## 9. Next steps

1. **Refine lagoon polygon** against v02 mesh + coastline (cell 2.draft is rectangular).
2. **Verify v02 volume estimate** against literature (~12 km², mean depth 1 m → ~12 ×10⁶ m³).
3. **Download ERA5 evaporation `e`** via `dfm_tools` and rerun cell 4.
4. **Build v03 model** with:
   - Corrected hypersaline initial field (42 ppt inside, 37.5 outside — `feedback_salinity_bc.md`)
   - Passive tracer as recipe in section 6
   - ERA5 evaporation coupled
5. **Run v03 for ≥30 days** (residence may be weeks; 9-day run won't give a full decay).
6. **Activate section 7** post-processing on v03 output.
7. **Ensemble over wind scenarios** — rerun with summer sirocco / winter mistral / calm forcing; report the distribution of τ.
8. **Optional — CART age tracer** as Paper 1 add-on for full spatial age field.